# TRIBE v2 — Content A/B Test on a Digital Brain Twin

We run 3 variants of a LinkedIn/Twitter hook through Meta's **TRIBE v2** model to predict which one produces the strongest brain activation.

**Setup (Colab):**
1. **Runtime → Change runtime type → T4 GPU**
2. Accept [LLaMA 3.2-3B license](https://huggingface.co/meta-llama/Llama-3.2-3B)
3. Add `HF_TOKEN` in Colab Secrets (key icon, left sidebar)

## Step 1 — Install & Restart

Run the cell below, then **restart the runtime** (Runtime → Restart session) before continuing.

In [ ]:
!uv pip install "tribev2[plotting] @ git+https://github.com/facebookresearch/tribev2.git"

## Step 2 — Authenticate & Load Model

In [ ]:
import os
# Try Colab Secrets first
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass  # Set HF_TOKEN manually if not using Colab Secrets

from huggingface_hub import login
login(new_session=False)

from tribev2.demo_utils import TribeModel, download_file
from tribev2.plotting import PlotBrain
from pathlib import Path
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

CACHE_FOLDER = Path("./cache")

model = TribeModel.from_pretrained(
    "facebook/tribev2",
    cache_folder=CACHE_FOLDER,
)
plotter = PlotBrain(mesh="fsaverage5")
print("Model loaded!")

## Step 3 — Define 3 Hook Variants

| Variant | Strategy |
|---------|----------|
| **A (Storyteller)** | Narrative arc, CoComelon story, builds to payoff |
| **B (Provocative)** | Opens with a question, meta-twist ending |
| **C (Blunt)** | Shortest. Punchy. No fat. Drops the mic. |

In [ ]:
hooks = {
    "A_storyteller": """Three years ago, CoComelon used to A/B test their videos in front of ten to fifteen live babies. Whichever version held attention the longest, that's the one that shipped to millions. Sounds absurd. But it worked. They became the most watched channel on YouTube. Now Meta just open sourced a model trained on eleven hundred hours of brain scans from seven hundred people. You feed it video, audio, or text. It predicts how the human brain responds. Seventy thousand data points across the cortex. No scanner needed. The next generation of content won't be tested on audiences. It'll be tested on a digital twin of the human brain before a single person ever sees it.""",

    "B_provocative": """What if you could test how someone's brain reacts to your content before they ever see it? No focus groups. No A/B tests. No waiting for analytics. Meta just made that real. They open sourced TRIBE v2, a model that predicts human brain activity from any video, audio, or text. Trained on seven hundred brains. Eleven hundred hours of scans. Seventy thousand data points per prediction. I ran this post through it before publishing. The version that lit up your temporal and prefrontal cortex the most? That's the one you're reading right now.""",

    "C_blunt": """CoComelon tested videos on live babies. Meta just open sourced a way to test content on a digital copy of the human brain. TRIBE v2. Seven hundred brains scanned. Eleven hundred hours of data. Feed it any video, audio, or text, and it predicts your neural response across seventy thousand cortical data points. No scanner. No subjects. No waiting. I tested this post on it before you read it. Your brain was already predicted.""",
}

for name, text in hooks.items():
    print(f"  {name}: {len(text.split())} words")

## Step 4 — Run Brain Predictions (all 3 hooks)

For each hook: Text → gTTS speech → WhisperX transcription → LLaMA 3.2 + Wav2Vec-BERT features → fMRI prediction

**~2-5 min per hook on T4 GPU**

In [ ]:
import numpy as np
import time

results = {}
all_segments = {}

for name, text in hooks.items():
    print(f"\n{'='*60}")
    print(f"Predicting: {name}")
    print(f"{'='*60}")
    start = time.time()

    text_path = CACHE_FOLDER / f"hook_{name}.txt"
    text_path.write_text(text)

    df = model.get_events_dataframe(text_path=text_path)
    preds, segments = model.predict(events=df)
    results[name] = preds
    all_segments[name] = segments

    elapsed = time.time() - start
    print(f"  Shape: {preds.shape}")
    print(f"  Mean: {np.mean(preds):.4f} | Peak: {np.max(preds):.4f}")
    print(f"  Time: {elapsed:.1f}s")

print(f"\n{'='*60}")
print("All 3 hooks predicted!")
print(f"{'='*60}")

## Step 5 — Brain Maps for Each Hook

Predicted fMRI activity on the cortical surface. Each row = 1 second of predicted brain response.

In [ ]:
for name, preds in results.items():
    print(f"\n{'='*60}")
    print(f"Brain Maps: {name}")
    print(f"{'='*60}")
    n = min(15, preds.shape[0])
    fig = plotter.plot_timesteps(
        preds[:n],
        segments=all_segments[name][:n],
        cmap="fire",
        norm_percentile=99,
        vmin=0.6,
        alpha_cmap=(0, 0.2),
        show_stimuli=True,
    )

## Step 6 — Activation Timeline Comparison

In [ ]:
import matplotlib.pyplot as plt

colors = {"A_storyteller": "#2196F3", "B_provocative": "#FF5722", "C_blunt": "#4CAF50"}
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Mean activation over time
for name, preds in results.items():
    axes[0].plot(np.mean(preds, axis=1), label=name, linewidth=2, color=colors[name])
axes[0].set_xlabel("Time (s)"); axes[0].set_ylabel("Mean Activation")
axes[0].set_title("Mean Brain Activation", fontweight="bold"); axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)

# Peak activation over time
for name, preds in results.items():
    axes[1].plot(np.max(preds, axis=1), label=name, linewidth=2, color=colors[name])
axes[1].set_xlabel("Time (s)"); axes[1].set_ylabel("Peak Activation")
axes[1].set_title("Peak Brain Activation", fontweight="bold"); axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)

# % strongly active vertices
for name, preds in results.items():
    thresh = np.mean(preds) + np.std(preds)
    axes[2].plot(np.mean(preds > thresh, axis=1) * 100, label=name, linewidth=2, color=colors[name])
axes[2].set_xlabel("Time (s)"); axes[2].set_ylabel("% Active Vertices")
axes[2].set_title("Brain Engagement", fontweight="bold"); axes[2].legend(fontsize=9); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("activation_timeline.png", dpi=200, bbox_inches="tight")
plt.show()
print("Saved: activation_timeline.png")

## Step 7 — Final Verdict

In [ ]:
import pandas as pd

rows = []
for name, preds in results.items():
    top5 = np.percentile(preds, 95)
    rows.append({
        "Hook": name,
        "Mean": f"{np.mean(preds):.4f}",
        "Peak": f"{np.max(preds):.4f}",
        "Spread": f"{np.std(preds):.4f}",
        "Strong %": f"{np.mean(preds > (np.mean(preds) + np.std(preds))) * 100:.1f}%",
        "Top 5% Intensity": f"{np.mean(preds[preds > top5]):.4f}",
    })

df_summary = pd.DataFrame(rows)
winner = max(results, key=lambda k: np.mean(results[k][results[k] > np.percentile(results[k], 95)]))

print("=" * 70)
print("FINAL RESULTS — TRIBE v2 Brain Activation A/B Test")
print("=" * 70)
display(df_summary)
print(f"\nWINNER: {winner}")
print(f"\nScreenshot these results for your LinkedIn post!")